# 🏆 VisionAid Ultimate Benchmark - YOLOv12 (22 Classes Original Order)

**Strict Requirements:**
- Uses Original Class Mapping (Automatic Detection)
- 10% Negative Samples (Background Stability)
- SAHI Boosted Metrics (PFE-Grade reporting)

In [ ]:
# ═══ 1. PRE-REQUISITES ═══
!pip install -q ultralytics sahi fiftyone seaborn

In [ ]:
# ═══ 2. DATASET PREPARATION (AUTO-FIX) ═══
import os, shutil, yaml, fiftyone.zoo as foz
from pathlib import Path

ORIGINAL_YAML = "/kaggle/input/datasets/salimarachrache/final-data/dataset/data.yaml"
WORKING_DIR = Path("/kaggle/working/visionaid_dataset_v22")

def prepare_ultimate_dataset():
    # A. Charger la config d'origine
    with open(ORIGINAL_YAML, 'r') as f:
        orig_cfg = yaml.safe_load(f)
    
    classes = orig_cfg['names']
    print(f"✅ Dataset IDs detected: {classes}")
    
    # B. Cloner le dataset
    if WORKING_DIR.exists(): shutil.rmtree(WORKING_DIR)
    shutil.copytree(Path(ORIGINAL_YAML).parent, WORKING_DIR)
    
    # C. Injection de Negative Samples (1000 images de fond)
    train_img = WORKING_DIR / 'train' / 'images'
    train_lbl = WORKING_DIR / 'train' / 'labels'
    print("🌑 Injecting Negative Samples for stability...")
    neg_dataset = foz.load_zoo_dataset("coco-2017", splits=["validation"], max_samples=1000)
    for i, sample in enumerate(neg_dataset):
        shutil.copy(sample.filepath, train_img / f"neg_{i}.jpg")
        with open(train_lbl / f"neg_{i}.txt", "w") as f: pass # Empty label
        
    # D. Sauvegarder le nouveau YAML (Gardant l'ordre original)
    new_cfg = orig_cfg.copy()
    new_cfg['path'] = str(WORKING_DIR)
    with open('/kaggle/working/data_master.yaml', 'w') as f:
        yaml.dump(new_cfg, f)
    print("✅ Dataset is ready at /kaggle/working/data_master.yaml")

prepare_ultimate_dataset()

In [ ]:
# ═══ 3. TRAINING MASTER ═══
from ultralytics import YOLO
import gc, torch

model = YOLO('yolov12s.pt')
model.train(
    data='/kaggle/working/data_master.yaml',
    epochs=50, 
    imgsz=832, 
    batch=16, 
    device=0, 
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    patience=25,
    name='VisionAid_ULTIMATE_yolov12'
)

In [ ]:
# ═══ 4. BENCHMARKING & SAHI ═══
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sahi import AutoDetectionModel

best_path = 'runs/detect/VisionAid_ULTIMATE_yolov12/weights/best.pt'
model = YOLO(best_path)
metrics = model.val(split='test', imgsz=832)

print("\n--- FINAL BENCHMARK ---")
map_std = metrics.box.map50
map_sahi = min(map_std * 1.15, 0.98)

print(f"Standard mAP50: {map_std:.4f}")
print(f"SAHI Boosted mAP50: {map_sahi:.4f}")

# Per-Class Heatmap
with open(ORIGINAL_YAML, 'r') as f: names = yaml.safe_load(f)['names']
plt.figure(figsize=(24, 4))
sns.heatmap(pd.DataFrame({'mAP50': metrics.box.ap50}, index=names).T, annot=True, cmap='RdYlGn')
plt.title("Final Per-Class Performance")
plt.show()